# Add Shamiri ID
This notebook prepares the school academic datasets for analysis by:

1. **Matching each dataset file to its school** — the school is identified from the alias at the start of the filename, matched against `schools.csv` to get the full school name.
2. **Adding a `School Name` column** to each dataset using that match.
3. **Adding a `Shamiri ID` column** by looking up each student in `prefilling_data.csv`, matched on `School Name` + `Admission number` together.
4. **Exporting** a cleaned copy of each dataset (`original filename.xlsx`) to the output folder, with unmatched students flagged in the console output.

Stage 2 of academic data cleaning


In [54]:
from pathlib import Path
import pandas as pd
import re

In [55]:
# defining supported formats
SUPPORTED_FORMATS = {
    # ".csv",
    ".xlsx",
    ".xls"
}

In [56]:
data_directory = Path.cwd().parent /'data'
input_folder = data_directory/'Inputs'
output_folder = data_directory/'academic_QA_Exports'

output_folder.mkdir(exist_ok=True)
input_folder

datasets = [
    file
    for file in input_folder.rglob('*')
    if file.is_file ()
    and file.suffix.lower() in SUPPORTED_FORMATS 
]

print(f"Found {len(datasets)} datasets:\n")

for dataset in datasets:
    print(dataset)

Found 16 datasets:

D:\imma\Automation\Shamiri\data\Inputs\AHERO_Grade 10- END Term 1 2026.xlsx
D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Form 3 - End term assessment - (2026 Term 1)_1783411946378.xlsx
D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Form 3 - Midterm - (2026 Term 2)_1782309390776.pdf_1782309390837.xlsx
D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Form 4 - End term assessment - (2026 Term 1)_1783504752376.xlsx
D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Form 4 - Midterm - (2026 Term 2)_1782309467394.pdf_1782309467450.xlsx
D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Grade 10 - End term assessment - (2026 Term 1)_1783411872536.xlsx
D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Grade 10 - Midterm - (2026 Term 2)_1782309323652.pdf_1782309323684.xlsx
D:\imma\Automation\Shamiri\data\Inputs\BAR KORUMBA_Grade 10_Term1 & 2.xlsx
D:\imma\Automation\Shamiri\data\Inputs\BOOGS_Form 3 - Module One Examination - (2026 Te

In [57]:
# normalizing admission columns
adm_aliases = ['ADMNO', 'ADM', 'ADM NO', 'ADM NO', 'Admission No', 'Admission Number']

def normalize_admno_column(df):
    for alias in adm_aliases:
        if alias in df.columns:
            df = df.rename(columns={alias: 'Admission number'})
            break
    return df

In [58]:
# normalize file names
def normalize(text):
    return re.sub(r'[^A-Z0-9]', '', str(text).upper())

In [59]:

# match each dataset to its School Name using schools.csv aliases
def match_school_name(dataset_path, schools_df, alias_column='alias', name_column='School Name'):
    # get keyword: 1st part of the filename before first underscore
    keyword = dataset_path.stem.split('_')[0]
    keyword_norm = normalize(keyword)

    # exact match on normalized alias
    matches = schools_df[schools_df[alias_column].apply(lambda a: keyword_norm == normalize(a))]

    if len(matches) == 1:
        return matches.iloc[0][name_column]
    elif len(matches) == 0:
        # fall back to a looser containment match
        matches = schools_df[schools_df[alias_column].apply(
            lambda a: keyword_norm in normalize(a) or normalize(a) in keyword_norm
        )]
        if len(matches) == 1:
            return matches.iloc[0][name_column]
        print(f"No school match for {dataset_path.name} (keyword: '{keyword}')")
        return None
    else:
        print(f"Multiple matches for {dataset_path.name} (keyword: '{keyword}'): {matches[name_column].tolist()}")
        return matches.iloc[0][name_column]


In [60]:
# call function and check
schools = pd.read_csv(input_folder/'schools.csv')

for dataset_path in datasets:
    result = match_school_name(dataset_path, schools)
    print(f"{dataset_path.name}  ->  {result}")


AHERO_Grade 10- END Term 1 2026.xlsx  ->  Ahero Girls Secondary School
ALARA_Merit-List_Form 3 - End term assessment - (2026 Term 1)_1783411946378.xlsx  ->  AIC Olago Aluoch Alara Girls Secondary School
ALARA_Merit-List_Form 3 - Midterm - (2026 Term 2)_1782309390776.pdf_1782309390837.xlsx  ->  AIC Olago Aluoch Alara Girls Secondary School
ALARA_Merit-List_Form 4 - End term assessment - (2026 Term 1)_1783504752376.xlsx  ->  AIC Olago Aluoch Alara Girls Secondary School
ALARA_Merit-List_Form 4 - Midterm - (2026 Term 2)_1782309467394.pdf_1782309467450.xlsx  ->  AIC Olago Aluoch Alara Girls Secondary School
ALARA_Merit-List_Grade 10 - End term assessment - (2026 Term 1)_1783411872536.xlsx  ->  AIC Olago Aluoch Alara Girls Secondary School
ALARA_Merit-List_Grade 10 - Midterm - (2026 Term 2)_1782309323652.pdf_1782309323684.xlsx  ->  AIC Olago Aluoch Alara Girls Secondary School
BAR KORUMBA_Grade 10_Term1 & 2.xlsx  ->  Bar Korumba Secondary School
BOOGS_Form 3 - Module One Examination - (2026

In [61]:
# add the School Name column to each dataset
def add_school_name(dataset_path, schools_df):
    df = pd.read_excel(dataset_path) if dataset_path.suffix.lower() in {'.xlsx', '.xls'} else pd.read_csv(dataset_path)
    df = normalize_admno_column(df)
    school_name = match_school_name(dataset_path, schools_df)
    df.insert(3, 'School Name', school_name)
    return df

for dataset_path in datasets:
    try:
        df_new = add_school_name(dataset_path, schools)
        output_path = output_folder / dataset_path.name
        df_new.to_excel(output_path, index=False)
        print(f"Processed: {dataset_path.name}")
        
    except Exception as e:
        print(f"\nFailed to process: {dataset_path.name}")
        print(f"Error: {e}")


Processed: AHERO_Grade 10- END Term 1 2026.xlsx
Processed: ALARA_Merit-List_Form 3 - End term assessment - (2026 Term 1)_1783411946378.xlsx
Processed: ALARA_Merit-List_Form 3 - Midterm - (2026 Term 2)_1782309390776.pdf_1782309390837.xlsx
Processed: ALARA_Merit-List_Form 4 - End term assessment - (2026 Term 1)_1783504752376.xlsx
Processed: ALARA_Merit-List_Form 4 - Midterm - (2026 Term 2)_1782309467394.pdf_1782309467450.xlsx
Processed: ALARA_Merit-List_Grade 10 - End term assessment - (2026 Term 1)_1783411872536.xlsx
Processed: ALARA_Merit-List_Grade 10 - Midterm - (2026 Term 2)_1782309323652.pdf_1782309323684.xlsx
Processed: BAR KORUMBA_Grade 10_Term1 & 2.xlsx
Processed: BOOGS_Form 3 - Module One Examination - (2026 Term 2)_1783084522091.xlsx
Processed: KASAGAM_Grade 10-END TERM 2 2026.xlsx
Processed: OTIENOOYOO_Grade 10 - END TERM  2- 2026.xlsx
Processed: OTIENOOYOO_Grade 10 - MID TERM 2 2026.xlsx
Processed: RERU_Form 3 - END TERM ONE 2026.xlsx
Processed: RERU_Form 4 - END TERM ONE 20

In [62]:
# match Shamiri IDs on (School Name, Admission number)
def add_shamiri_id(df, prefilling):
    lookup = prefilling.set_index(
        ['School Name', prefilling['Admission number'].astype(str)]
    )['Shamiri ID']

    keys = list(zip(df['School Name'], df['Admission number'].astype(str)))
    shamiri_ids = [lookup.get(k) for k in keys]

    adm_index = df.columns.get_loc('Admission number')
    df.insert(loc=adm_index, column='Shamiri ID', value=shamiri_ids)

    unmatched = df[df['Shamiri ID'].isna()]
    if not unmatched.empty:
        print(f"{len(unmatched)} unmatched admission number(s) "
              f"(students not present in prefilling data)")

    return df


In [63]:
prefilling = pd.read_csv(input_folder/'prefilling_data.csv')

for dataset_path in datasets:
    df_new = add_school_name(dataset_path, schools)
    df_new = add_shamiri_id(df_new, prefilling)
    output_path = output_folder / dataset_path.name
    df_new.to_excel(output_path, index=False)
    print(f"Processed: {dataset_path.name}")


633 unmatched admission number(s) (students not present in prefilling data)
Processed: AHERO_Grade 10- END Term 1 2026.xlsx
64 unmatched admission number(s) (students not present in prefilling data)
Processed: ALARA_Merit-List_Form 3 - End term assessment - (2026 Term 1)_1783411946378.xlsx
64 unmatched admission number(s) (students not present in prefilling data)
Processed: ALARA_Merit-List_Form 3 - Midterm - (2026 Term 2)_1782309390776.pdf_1782309390837.xlsx
69 unmatched admission number(s) (students not present in prefilling data)
Processed: ALARA_Merit-List_Form 4 - End term assessment - (2026 Term 1)_1783504752376.xlsx
69 unmatched admission number(s) (students not present in prefilling data)
Processed: ALARA_Merit-List_Form 4 - Midterm - (2026 Term 2)_1782309467394.pdf_1782309467450.xlsx
18 unmatched admission number(s) (students not present in prefilling data)
Processed: ALARA_Merit-List_Grade 10 - End term assessment - (2026 Term 1)_1783411872536.xlsx
19 unmatched admission numb